# 06. Model comparison

Este notebook recupera runs ya registrados en MLflow para comparar los cuatro modelos didácticos de `invoice-risk-v1`. No genera datos ni entrena modelos.

## Configuración requerida

Configure MLflow como se documenta en el README mediante `MLFLOW_TRACKING_URI`. Use el mismo Workspace y contexto académico con los que se registraron los runs. No escriba URLs ni credenciales en este notebook.

In [ ]:
import mlflow
from mlflow.entities import ViewType

from invoiceops_ml.mlflow import configure_mlflow, mlflow_config_from_env

MODEL_RUN_NAMES = (
    'dummy-baseline',
    'logistic-regression',
    'random-forest',
    'hist-gradient-boosting',
)
SUMMARY_METRICS = ('validation_accuracy', 'validation_precision', 'validation_recall', 'validation_f1')
DECISION_METRICS = ('validation_recall', 'validation_precision', 'validation_f1')
OWNERSHIP_TAGS = ('organization_slug', 'owner_type', 'owner_id', 'created_by_rut')

configure_mlflow(mlflow_config_from_env())
client = mlflow.MlflowClient()
experiment_ids = [experiment.experiment_id for experiment in client.search_experiments()]
runs = []
page_token = None
while True:
    page = client.search_runs(
        experiment_ids=experiment_ids,
        run_view_type=ViewType.ACTIVE_ONLY,
        page_token=page_token,
    )
    runs.extend(page)
    if not page.token:
        break
    page_token = page.token

comparison_rows = [
    {
        'run_id': run.info.run_id,
        'run_name': run.data.tags.get('mlflow.runName'),
        **{tag: run.data.tags.get(tag) for tag in OWNERSHIP_TAGS},
        **{metric: run.data.metrics.get(metric) for metric in SUMMARY_METRICS},
    }
    for run in runs
    if run.data.tags.get('mlflow.runName') in MODEL_RUN_NAMES
]

missing_run_names = [
    run_name
    for run_name in MODEL_RUN_NAMES
    if run_name not in {row['run_name'] for row in comparison_rows}
]
if missing_run_names:
    raise ValueError(
        f"Missing required model runs: {', '.join(missing_run_names)}. "
        'Run notebooks 02 through 05 first.'
    )

comparison_rows

## Compare Runs en MLflow UI

Priorice la UI de MLflow para comparar runs: abra el experimento, filtre por los cuatro `run_name`, seleccione los runs y use **Compare**. Revise parámetros, tags de ownership y las métricas `validation_*` lado a lado. La tabla de abajo es un apoyo reproducible para registrar la decisión, no reemplaza la revisión visual de la UI.

In [ ]:
def rank_candidates(rows: list[dict[str, object]]) -> list[dict[str, object]]:
    complete_rows = [
        row
        for row in rows
        if all(row[metric] is not None for metric in DECISION_METRICS)
    ]
    return sorted(
        complete_rows,
        key=lambda row: (
            float(row['validation_recall']),
            float(row['validation_precision']),
            float(row['validation_f1']),
        ),
        reverse=True,
    )


ranked_candidates = rank_candidates(comparison_rows)
if not ranked_candidates:
    raise ValueError(
        'Model runs must include validation_recall, validation_precision, and validation_f1 '
        'before comparison.'
    )

candidate = ranked_candidates[0]
candidate

## Candidate decision

El candidate sugerido prioriza `validation_recall`, luego `validation_precision` y `validation_f1`; por eso la selección no usa métricas de test. Antes de aceptarlo, justifique la elección con el riesgo de negocio de omitir una factura riesgosa, los trade-offs observados en la UI y los tags de ownership. Registre el `run_id` y la justificación en el trabajo del curso. Este notebook no ejecuta un Gate ni promueve modelos; ambas tareas pertenecen a ML-10 y ML-13.